# Pokémon Dataset — Extraction & Exploratory Data Analysis
**Phase 1 (Extraction) and Phase 2 (EDA) of the ETL pipeline**

## Dataset & sources
| source | file | rows |
|---|---|---|
| Original raw CSV (gens 1–7, dex 1–801) | `datasets/pokemon.csv` | 801 |
| PokeAPI (dex 802–1025) | cached JSON in `datasets/raw/` | 224 |
| Extended raw dataset used below | `datasets/pokemon_full.csv` | 1025 |

* `datasets/pokemon.csv` is the **raw source and is never modified** — every
  transformation writes to new files.
* The 224 Pokémon of generations VIII & IX (plus 8 gen-7 entries missing from
  the original file) were extracted from [PokeAPI](https://pokeapi.co) with
  `scripts/fetch_pokeapi.py` and mapped onto the original 41 columns with
  `scripts/build_dataset.py`. Raw API responses are kept in `datasets/raw/`.
* The mapping was **validated first**: 73 dex entries that already exist in the
  CSV were rebuilt from the API and diffed column by column (see §2.6).

Reproduce everything from the project root:
```bash
python scripts/fetch_pokeapi.py      # extract raw JSON (cached on disk)
python scripts/build_dataset.py      # validate + build datasets/pokemon_full.csv
```

> Run this notebook from the project root so the relative paths resolve.

In [ ]:
%matplotlib inline
import ast
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_CSV = Path("datasets/pokemon.csv")
FULL_CSV = Path("datasets/pokemon_full.csv")
RAW_API = Path("datasets/raw")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

df = pd.read_csv(FULL_CSV)
print(f"{df.shape[0]} rows x {df.shape[1]} columns")
df.head(3)

In [ ]:
# the original raw file is still untouched
raw = pd.read_csv(RAW_CSV)
print(f"raw source      : {raw.shape[0]} rows x {raw.shape[1]} columns")
print(f"extended dataset: {df.shape[0]} rows x {df.shape[1]} columns "
      f"(+{df.shape[0] - raw.shape[0]} rows from PokeAPI)")
print("columns added by the extension:",
      sorted(set(df.columns) - set(raw.columns)) or "none - same 41 columns")
print("column order changed:", list(df.columns[:4]), "...")

## 1. Basic dataset exploration

Shape, column names, data types, sample records, summary statistics and
unique values — before any cleaning takes place.

In [ ]:
print("Rows:", df.shape[0], " Columns:", df.shape[1], "\n")
print("Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3d}. {col}")

In [ ]:
# data types + completeness + cardinality in one table
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(),
})
overview

In [ ]:
# sample records
display(df.head(4))
display(df.sample(5, random_state=42))

In [ ]:
# summary statistics (numeric columns only)
display(df.describe().T)
print("capture_rate is NOT in describe() because it is stored as text:")
df.capture_rate.head(3)

In [ ]:
# unique values in the important columns
for col in ["generation", "type1", "is_legendary"]:
    vals = df[col].value_counts().sort_index()
    print(f"{col} ({df[col].nunique()} unique): {vals.to_dict()}")

print("\nnumber of unique values in the key columns:")
print(df[["pokedex_number", "name", "japanese_name", "classfication",
          "abilities", "type2", "percentage_male"]].nunique())

In [ ]:
# all 18 types appear as primary type, none outside the official chart
OFFICIAL_TYPES = {"bug", "dark", "dragon", "electric", "fairy", "fighting",
                  "fire", "flying", "ghost", "grass", "ground", "ice",
                  "normal", "poison", "psychic", "rock", "steel", "water"}
print("unexpected type1 values:", set(df.type1) - OFFICIAL_TYPES or "none")
print("unexpected type2 values:", set(df.type2.dropna()) - OFFICIAL_TYPES or "none")
print("dual-type Pokémon:", int(df.type2.notna().sum()),
      f"({df.type2.notna().mean():.0%})  |  single-type:",
      int(df.type2.isna().sum()))

## 2. Data quality analysis

Missing values, duplicates, wrong data types, invalid/inconsistent values,
outliers, unexpected categories and formatting problems.

In [ ]:
# --- missing / null values ------------------------------------------------
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "of_total_%": (df.isna().mean() * 100).round(1),
}).query("missing > 0").sort_values("missing", ascending=False)
display(missing)

print("\ntype2 and percentage_male are *structural* blanks:",
      "\n  * a Pokémon without a second type has no type2,",
      "\n  * genderless Pokémon (Porygon, Magnemite, Ultra Beasts ...) have no ratio.",
      "\nheight_m / weight_kg blanks are a real defect - see section 2.4.")

In [ ]:
# --- duplicate records ----------------------------------------------------
print("fully duplicated rows :", int(df.duplicated().sum()))
print("duplicate names       :", int(df.duplicated('name').sum()))
print("duplicate dex numbers :", int(df.duplicated('pokedex_number').sum()))
print("dex range covered     :", int(df.pokedex_number.min()), "-",
      int(df.pokedex_number.max()),
      "| gaps:", sorted(set(range(1, 1026)) - set(df.pokedex_number)) or "none")

In [ ]:
# --- data types -----------------------------------------------------------
print(df.dtypes.value_counts(), "\n")
print("columns that look numeric but are stored as text:")
for col in df.columns:
    coerced = pd.to_numeric(df[col], errors="coerce")
    if df[col].dtype != "float64" and df[col].dtype != "int64" \
            and coerced.notna().sum() >= len(df) * 0.9:
        bad = df.loc[coerced.isna() & df[col].notna(), col]
        print(f"  {col}: {coerced.notna().sum()}/{len(df)} convertible -> "
              f"exceptions: {bad.unique()[:3].tolist()}")

In [ ]:
# abilities / japanese_name are stringified structures, not plain text
print("abilities example :", df.abilities.iloc[0])
print("parsed            :", ast.literal_eval(df.abilities.iloc[0]))
print("japanese_name     :", df.japanese_name.iloc[0], "= romaji + japanese script")

### 2.1 — `against_*` columns vs. the official type chart

The 18 `against_*` columns should equal *the damage an attacker of that type
deals to this Pokémon* — i.e. the type chart applied to **both** defensive
types (dual types multiply). The chart below is rebuilt from the cached
PokeAPI type data, so nothing is hard-coded.

In [ ]:
# rebuild the type chart from the raw API cache
chart = {}
for path in sorted(RAW_API.glob("type_*.json")):
    name = path.stem.split("_", 1)[1]
    rel = json.loads(path.read_text())["damage_relations"]
    multipliers = {d["name"]: 2.0 for d in rel["double_damage_to"]}
    multipliers.update({d["name"]: 0.5 for d in rel["half_damage_to"]})
    multipliers.update({d["name"]: 0.0 for d in rel["no_damage_to"]})
    chart[name] = multipliers


def expected_against(types):
    cells = {}
    for atk in chart:
        col = "against_fight" if atk == "fighting" else f"against_{atk}"
        value = 1.0
        for defence in types:          # dual types multiply
            value *= chart[atk].get(defence, 1.0)
        cells[col] = value
    return cells


problems = []
for row in df.itertuples():
    types = [row.type1] + ([row.type2] if isinstance(row.type2, str) else [])
    for col, expected in expected_against(types).items():
        actual = float(getattr(row, col))
        if abs(actual - expected) > 1e-9:
            problems.append([row.pokedex_number, row.name, col, actual, expected])

bad_against = pd.DataFrame(problems, columns=["dex", "name", "column", "csv_value", "type_chart_value"])
print(f"{len(bad_against)} mismatching cells across {bad_against.name.nunique()} Pokémon")
print(f"(checked {len(df) * 18} cells = 1025 Pokémon x 18 attacking types)")
bad_against.groupby("name").size().sort_values(ascending=False)

In [ ]:
# the CSV says these are dual-typed but gives single-type effectiveness values
display(bad_against[bad_against.name.isin(["Rattata", "Meowth", "Marowak"])].head(12))
# ... and the same rows carry regional (Alolan) types with base-form measurements
display(df.loc[df.name.isin(["Rattata", "Meowth", "Vulpix", "Raichu"]),
               ["pokedex_number", "name", "type1", "type2", "height_m", "weight_kg"]])

In [ ]:
# --- inconsistent / invalid values ----------------------------------------
# 1) type2 identical to type1 (impossible for a real Pokémon)
same = df[df.type2.notna() & (df.type1 == df.type2)]
print("type2 == type1:", same.name.tolist(), "\n")

# 2) percentage_male outside the values the game actually uses
STANDARD_RATIOS = {0.0, 12.5, 25.0, 37.5, 50.0, 62.5, 75.0, 87.5, 100.0}
odd_ratio = df[df.percentage_male.notna() & ~df.percentage_male.isin(STANDARD_RATIOS)]
print("non-standard gender ratios in the CSV:")
print(odd_ratio.percentage_male.value_counts().to_dict())
print("  -> the original file uses 88.1 / 75.4 / 24.6 / 11.2 where the games",
      "\n     use 87.5 / 75 / 25 / 12.5. Rows 1-801 are affected;",
      "\n     the PokeAPI rows use the correct values.")
print("rows with a non-standard value:", len(odd_ratio), "\n")

# 3) capture_rate should be an integer
not_int = df[~df.capture_rate.astype(str).str.fullmatch(r"\d+")]
print("capture_rate values that are not a plain integer:")
print(not_int[["pokedex_number", "name", "capture_rate"]].to_string(index=False))

In [ ]:
# --- unexpected categories ------------------------------------------------
print("is_legendary values :", sorted(df.is_legendary.unique()))
print("generation values    :", sorted(df.generation.unique()))
print("type1 values         :", df.type1.nunique(), "of 18 official types")
print("rows per generation  :", df.generation.value_counts().sort_index().to_dict())
print("\nlegendary flag also contains mythicals (Mew, Deoxys, Keldeo ...),",
      "\nwhich is how the original file was built:",
      df.loc[df.name.isin(["Mew", "Deoxys", "Keldeo"]), ["name", "is_legendary"]].to_string(index=False))

### 2.2 — Outliers (IQR method)

Values outside `Q1 − 1.5·IQR` / `Q3 + 1.5·IQR`. Most are legitimate — Pokémon
like Wailord and Celesteela are meant to be extreme — but they are worth
listing before deciding what to do with them.

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


numeric = ["height_m", "weight_kg", "hp", "attack", "defense",
           "sp_attack", "sp_defense", "speed", "base_total", "base_egg_steps"]
rows = []
for col in numeric:
    lo, hi = iqr_bounds(df[col])
    out = df[(df[col] < lo) | (df[col] > hi)]
    rows.append({"column": col, "low": lo, "high": hi,
                 "outliers": len(out),
                 "max example": df.loc[df[col].idxmax(), "name"] + f" ({df[col].max()})"})
outliers = pd.DataFrame(rows)
outliers

In [ ]:
# example: who are the extreme ones?
display(df.nlargest(5, "weight_kg")[["name", "weight_kg", "type1", "generation"]])
display(df.nlargest(5, "height_m")[["name", "height_m", "type1", "generation"]])
display(df.nlargest(5, "hp")[["name", "hp", "is_legendary"]])

### 2.3 — Formatting & consistency issues

Text fields that disagree with their own official spellings.

In [ ]:
# 1) ability names: compare every token with the official ability index
idx = json.loads((RAW_API / "ability_index.json").read_text())
official = {r["name"] for r in idx["results"]}          # slug form, e.g. compound-eyes
by_norm = {s.replace("-", "").replace(" ", ""): s for s in official}

spelling, whitespace = {}, set()
for value in df.abilities:
    for token in ast.literal_eval(value):
        if token != token.strip():
            whitespace.add(repr(token))
        candidate = by_norm.get(token.strip().replace("-", "").replace(" ", "").lower())
        if candidate and token.strip().lower() not in {candidate,
                                                       candidate.replace("-", " ").lower()}:
            spelling[token] = candidate
print("ability names that differ from the official spelling:", spelling)
print("ability names with stray whitespace:", whitespace or "none")

# 2) columns with typos / free-form labels
print("\nmisspelled column name:", [c for c in df.columns if c == "classfication"])
print("names with punctuation:", df.loc[df.name.str.contains(r"[^A-Za-z0-9 \- ']"), "name"].tolist()[:12])
print("japanese_name entries with a form label:",
      df.japanese_name.str.contains("\(").sum(), "rows")

In [ ]:
# 3) rows whose height/weight are missing although every Pokémon has both
missing_size = df[df.height_m.isna() | df.weight_kg.isna()]
print(f"{len(missing_size)} rows without height/weight:")
missing_size[["pokedex_number", "name", "type1", "type2", "generation"]].to_string(index=False)

### 2.4 — Validation of the PokeAPI field mapping

`scripts/build_dataset.py` rebuilt 73 dex entries that already exist in the raw
CSV and diffed every column. The remaining differences are **defects in the
original file**, not in the mapping — this is the list Phase 3 has to clean:

| column | finding |
|---|---|
| `percentage_male` | 15 rows use 88.1 instead of 87.5 (and similar) |
| `attack`, `sp_attack`, `base_total`, ... | mega/regional-form stats merged into the base row (Charizard `634` = Mega Charizard X) |
| `type2`, `height_m`, `weight_kg` | Alolan-form rows: duplicated `type2`, empty size |
| `abilities` | form-specific abilities (`Surge Surfer`) and bad spellings (`Compoundeyes`) |
| `classfication`, `japanese_name` | free-form wording / form labels |
| `capture_rate` | `30 (Meteorite)255 (Core)` (Minior), Necrozma `3` vs API `255` |

Re-run it any time with `python scripts/build_dataset.py`.

## 3. Exploratory analysis

Distribution of Pokémon across generations and types, base-stat trends,
legendaries, physical vs. special attackers, size, catch rates and abilities.

In [ ]:
# --- Pokémon per generation ----------------------------------------------
ax = df.generation.value_counts().sort_index().plot(
    kind="bar", figsize=(9, 4), color="steelblue", edgecolor="black")
ax.set_title("Pokémon introduced per generation")
ax.set_xlabel("generation")
ax.set_ylabel("number of Pokémon")
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()

In [ ]:
# --- primary / secondary type distribution --------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df.type1.value_counts().plot(kind="bar", ax=axes[0], color="coral", edgecolor="black")
axes[0].set_title("Primary type (type1)")
axes[0].set_ylabel("number of Pokémon")
axes[0].tick_params(axis="x", rotation=45)

df.type2.value_counts().plot(kind="bar", ax=axes[1], color="lightsteelblue", edgecolor="black")
axes[1].set_title("Secondary type (type2) — only dual-type Pokémon")
axes[1].set_ylabel("number of Pokémon")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()

In [ ]:
# --- most common type combinations ----------------------------------------
df["type_pair"] = df.type1 + " / " + df.type2.fillna("(mono-type)")
ax = df.type_pair.value_counts().head(12).sort_values().plot(
    kind="barh", figsize=(8, 6), color="seagreen", edgecolor="black")
ax.set_title("Most common type combinations")
ax.set_xlabel("number of Pokémon")
plt.tight_layout()

In [ ]:
# --- base stat total: distribution and trend over generations -------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

df.base_total.hist(bins=30, ax=axes[0], color="mediumpurple", edgecolor="black")
axes[0].axvline(df.base_total.mean(), color="black", ls="--",
                label=f"mean {df.base_total.mean():.0f}")
axes[0].set_title("Distribution of base stat total")
axes[0].set_xlabel("base_total")
axes[0].legend()

df.boxplot(column="base_total", by="generation", ax=axes[1])
axes[1].set_title("Base stat total by generation")
axes[1].set_xlabel("generation")
axes[1].set_ylabel("base_total")
fig.suptitle("")
plt.tight_layout()

In [ ]:
# average power creep per generation
by_gen = df.groupby("generation").agg(
    pokemons=("name", "count"),
    avg_base_total=("base_total", "mean"),
    avg_hp=("hp", "mean"),
    avg_speed=("speed", "mean"),
).round(1)
by_gen

In [ ]:
# --- which primary type is the strongest? ---------------------------------
by_type = (df.groupby("type1")
             .agg(pokemons=("name", "count"),
                  avg_base_total=("base_total", "mean"),
                  avg_attack=("attack", "mean"),
                  avg_sp_attack=("sp_attack", "mean"))
             .sort_values("avg_base_total", ascending=False)
             .round(1))
display(by_type)

ax = by_type.avg_base_total.plot(kind="bar", figsize=(10, 4),
                                 color="slateblue", edgecolor="black")
ax.axhline(df.base_total.mean(), color="black", ls="--",
           label=f"overall mean {df.base_total.mean():.0f}")
ax.set_title("Average base stat total by primary type")
ax.set_ylabel("base_total")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()

In [ ]:
# --- strongest Pokémon -----------------------------------------------------
top = df.nlargest(10, "base_total")[
    ["pokedex_number", "name", "type1", "type2", "generation", "base_total", "is_legendary"]]
ax = top.set_index("name").base_total.plot(kind="bar", figsize=(9, 4),
                                           color="crimson", edgecolor="black")
ax.set_title("Top 10 Pokémon by base stat total")
ax.set_ylabel("base_total")
ax.tick_params(axis="x", rotation=45)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
top

In [ ]:
# --- legendary vs. regular -------------------------------------------------
compare = df.groupby("is_legendary")[["hp", "attack", "defense",
                                      "sp_attack", "sp_defense", "speed",
                                      "base_total"]].mean().round(1)
compare.index = ["regular", "legendary/mythical"]
display(compare)
print(f"legendary share: {df.is_legendary.mean():.1%}  ({int(df.is_legendary.sum())} Pokémon)")
print("legendary average base stat total is",
      f"{compare.loc['legendary/mythical', 'base_total'] / compare.loc['regular', 'base_total']:.2f}x",
      "the regular average")

In [ ]:
# --- calculated column: physical or special attacker ----------------------
df["main_offense"] = np.where(df.attack > df.sp_attack, "physical",
                       np.where(df.sp_attack > df.attack, "special", "balanced"))
print(df.main_offense.value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
df.main_offense.value_counts().plot(kind="pie", ax=axes[0], autopct="%1.0f%%",
                                    colors=["navy", "salmon", "gray"], startangle=90)
axes[0].set_title("Attacking style")
axes[0].set_ylabel("")

for style, group in df.groupby("main_offense"):
    axes[1].scatter(group.attack, group.speed, s=14, alpha=0.6, label=style)
axes[1].set_xlabel("attack")
axes[1].set_ylabel("speed")
axes[1].set_title("Attack vs speed by attacking style")
axes[1].legend()
plt.tight_layout()

In [ ]:
# --- correlations ----------------------------------------------------------
num_cols = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed",
            "base_total", "height_m", "weight_kg"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(num_cols)), corr.columns)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="correlation")
ax.set_title("Correlation between numeric attributes")
plt.tight_layout()

print("strongest correlations with base_total:")
print(corr.base_total.drop("base_total").sort_values(ascending=False).round(2).to_string())

In [ ]:
# --- size: height vs weight ------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6))
for gen, group in df.groupby("generation"):
    ax.scatter(group.height_m, group.weight_kg, s=18, alpha=0.65, label=f"gen {gen}")
ax.set_xlabel("height (m)")
ax.set_ylabel("weight (kg)")
ax.set_title("Height vs weight, coloured by generation")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()

print("heaviest :", df.nlargest(3, "weight_kg")[["name", "weight_kg"]].values.tolist())
print("tallest  :", df.nlargest(3, "height_m")[["name", "height_m"]].values.tolist())
print("smallest :", df.nsmallest(3, "weight_kg")[["name", "weight_kg"]].values.tolist())

In [ ]:
# --- catch rate: legendary vs regular -------------------------------------
rate = pd.to_numeric(df.capture_rate, errors="coerce")   # Minior's mixed value -> NaN
tmp = pd.DataFrame({"capture_rate": rate, "is_legendary": df.is_legendary})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
tmp.boxplot(column="capture_rate", by="is_legendary", ax=axes[0])
axes[0].set_xticklabels(["regular", "legendary/mythical"])
axes[0].set_title("Catch rate by legendary status")
axes[0].set_xlabel("")
rate.hist(bins=30, ax=axes[1], color="goldenrod", edgecolor="black")
axes[1].set_title("Catch rate distribution")
axes[1].set_xlabel("capture_rate")
fig.suptitle("")
plt.tight_layout()

print("rows where capture_rate could not be parsed:")
df.loc[rate.isna(), ["pokedex_number", "name", "capture_rate"]].to_string(index=False)

In [ ]:
# --- gender ratios ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
df.percentage_male.plot(kind="hist", bins=25, ax=ax, color="teal", edgecolor="black")
ax.axvline(50, color="black", ls=":", label="50% male")
ax.set_title("Distribution of percentage_male")
ax.set_xlabel("percentage_male")
ax.legend()
print("genderless rows (blank):", int(df.percentage_male.isna().sum()))
print("value counts:", df.percentage_male.value_counts().sort_index().to_dict())

In [ ]:
# --- most common abilities -------------------------------------------------
counter = Counter()
for value in df.abilities:
    counter.update(ast.literal_eval(value))
abilities = pd.Series(counter, name="count").sort_values(ascending=False)

print(f"{len(abilities)} distinct abilities over {len(df)} Pokémon")
ax = abilities.head(15).sort_values().plot(kind="barh", figsize=(8, 6),
                                           color="mediumvioletred", edgecolor="black")
ax.set_title("Most common abilities")
ax.set_xlabel("times it appears in a Pokémon's ability list")
plt.tight_layout()
abilities.head(10)

## 4. EDA summary

### Exploratory findings
1. **Generations VIII & IX added 216 Pokémon** (96 + 120), taking the national
   Pokédex to 1025; generation V is still the largest single generation (156),
   followed by IX (120).
2. **Water (134) and Normal (118) are the most common primary types**, Flying
   is almost always a secondary type (9 as `type1`), and 52% of all Pokémon
   are dual-typed.
3. **Average base stat total rises with the generations** — from ≈413 (gen II)
   to ≈457 (gen IX) — a clear "power creep"; dragon and steel are on average
   the strongest primary types.
4. **Legendary/mythical Pokémon (98, 9.6%) average ~602 base stats vs ~416
   for regular Pokémon**, with every stat category higher.
5. **`base_total` tracks the offensive stats** (attack 0.73, sp_attack 0.72,
   sp_defense 0.71), attack correlates only weakly with speed (0.36), and size
   has a moderate — not decisive — link to power (height 0.50 / weight 0.45).
6. **Catch rate cleanly separates legendaries** (mostly ≤10) from regular
   Pokémon, and `Sturdy`/`Swift Swim`/`Keen Eye` are the most common abilities.

### Data-quality issues found (input for Phase 3 cleaning)
| # | issue | scope |
|---|---|---|
| 1 | `percentage_male` uses 88.1 / 75.4 / 24.6 / 11.2 instead of 87.5 / 75 / 25 / 12.5 | 156 rows |
| 2 | mega/regional-form stats merged into base rows (e.g. Charizard `base_total` 634 = Mega Charizard, Alolan Raichu stats) | several rows in gens 1–7 |
| 3 | Alolan-form data mixed in: duplicated `type2`, empty `height_m`/`weight_kg` | 20 rows |
| 4 | `against_*` disagrees with `type1`/`type2` (form rows) | 95 cells / 16 Pokémon |
| 5 | `capture_rate` stored as text with a mixed value (`30 (Meteorite)255 (Core)`) | 1 column, 1 row |
| 6 | ability spelling (`Compoundeyes`, `Lightningrod`, trailing space) | few rows |
| 7 | column name typo `classfication`, `japanese_name` form labels | 2 columns |
| 8 | `type2` structurally blank for 483 single-type rows, `percentage_male` blank for 155 genderless | expected, keep as null |
| 9 | outliers (Wailord/Celesteela size, Blissey HP) | legitimate, document |